# Phase 2 — Manifests and patient-grouped splits

Turns the 512 px cache plus each dataset's labels into the manifests everything
downstream reads, then assembles the development variants and the locked externals.

## Notebook settings (right-hand panel)

| Setting | Value |
|---|---|
| Accelerator | **None** — no GPU quota used |
| Persistence | No persistence |
| Internet | **On** (git clone) |
| Environment | **Pin to original environment** |

## Inputs to add

**The cache, *and* the raw datasets.** Labels live in the raw mounts, not the cache —
DDR's `train/valid/test.txt`, IDRiD's Part B grading CSV and Part C coordinate tables,
APTOS's `train.csv`, Messidor-2's grades. IDRiD coordinate re-projection also needs the
original Part A images, because the centres are published in original pixel space and
the geometry has to be recomputed.

From Phase 3 onward the cache alone is enough. Phase 2 is the last time the raw
sources are needed.

## Exit condition

`dataset_plan.json` written, the patient-disjointness assertion passing, and EyePACS
reporting close to **2.0 images per patient**. Anything near 1.0 means patient parsing
failed and every downstream number would be inflated.

## 1 · Pull the code

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"
print("repo ready at", REPO_DIR)

In [ ]:
from pathlib import Path
import os, re, json

INPUT = Path("/kaggle/input")

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace("%20", "")

def dataset_roots(max_depth=3):
    """Candidate dataset directories, shallowest first.

    Kaggle does not always mount datasets as direct children of /kaggle/input --
    they can sit under competitions/ and datasets/ wrappers. Breadth-first so a
    shallower match always wins over a nested subfolder of the same name.
    """
    level, out = [INPUT], []
    for _ in range(max_depth):
        nxt = []
        for d in level:
            try:
                children = sorted(c for c in d.iterdir() if c.is_dir())
            except OSError:
                continue
            out.extend(children)
            nxt.extend(children)
        level = nxt
    return out

EXCLUDE_ROOTS = []      # set to [CACHE] once the cache is located

def _under(path, root):
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False

def find_mount(*keywords, required=True):
    """Locate a mounted dataset by keyword, so a renamed mirror doesn't break the notebook.

    Anything under EXCLUDE_ROOTS is skipped. The cache contains directories named
    ddr, eyepacs, idrid, aptos and messidor2 -- exactly the keywords searched for --
    so without this a raw-dataset lookup can land inside the cache instead.
    """
    for d in dataset_roots():
        if any(_under(d, r) for r in EXCLUDE_ROOTS):
            continue
        if all(_norm(k) in _norm(d.name) for k in keywords):
            return d
    if required:
        print(f"  !! NOT MOUNTED: {' + '.join(keywords)}")
        print("     Use 'Add Input' in the right-hand panel. Candidate dirs seen:")
        for d in dataset_roots(2)[:25]:
            print(f"       - {d.relative_to(INPUT)}")
    return None

def find_any(*keyword_sets, label="", required=True):
    """Try several keyword spellings. Mirrors title themselves inconsistently:
    IDRiD ships as 'idrid-dataset' or 'indian-diabetic-retinopathy-image-dataset'."""
    for kws in keyword_sets:
        hit = find_mount(*kws, required=False)
        if hit:
            return hit
    if required:
        print(f"  !! NOT MOUNTED: {label or keyword_sets[0]}")
        print("     Currently mounted:")
        for d in sorted(INPUT.iterdir()):
            print(f"       - {d.name}")
    return None

def match_channel(dirname):
    """Map a mask directory name to a lesion channel.

    DDR uses MA/HE/EX/SE; IDRiD uses '1. Microaneurysms', '2. Haemorrhages',
    '3. Hard Exudates', '4. Soft Exudates', '5. Optic Disc'. Match on meaning so
    one function covers both.
    """
    n = re.sub(r"^\d+\.\s*", "", dirname.lower().strip())
    n = n.replace("%20", " ")
    if n == "ma" or "microaneurysm" in n:
        return "microaneurysm"
    if n == "he" or "haemorrhage" in n or "hemorrhage" in n:
        return "haemorrhage"
    if n == "ex" or ("hard" in n and "exudate" in n):
        return "hard_exudate"
    if n == "se" or ("soft" in n and "exudate" in n) or "cotton" in n:
        return "soft_exudate"
    if n == "od" or "optic disc" in n:
        return "optic_disc"
    return None

MASK_EXT = {".tif", ".tiff", ".png", ".gif", ".bmp", ".jpg", ".jpeg"}
LESION4 = ["microaneurysm", "haemorrhage", "hard_exudate", "soft_exudate"]

def scan_mask_dirs(root, require=""):
    """Find mask directories under root, keyed by lesion channel."""
    found = {}
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if require and require not in str(d).lower():
            continue
        channel = match_channel(d.name)
        if not channel:
            continue
        files = [f for f in d.rglob("*") if f.suffix.lower() in MASK_EXT]
        if files:
            found.setdefault(channel, []).append((str(d), len(files)))
    return found

def du(path, cap=40000):
    """Rough size + file count, capped so it stays fast on huge mounts."""
    total = n = 0
    for i, f in enumerate(Path(path).rglob("*")):
        if i > cap:
            return total, n, True
        if f.is_file():
            total += f.stat().st_size
            n += 1
    return total, n, False

print("helpers ready")

In [ ]:
import shlex, subprocess, time, zipfile

MANIFESTS = Path("/kaggle/working/manifests")
MANIFESTS.mkdir(parents=True, exist_ok=True)

def cache_candidates():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. Searched in /kaggle/input first: a stale
    copy left in /kaggle/working must never silently win over the published
    dataset you meant to use.
    """
    found = []
    for base in (INPUT, Path("/kaggle/working")):
        if not base.exists():
            continue
        roots = Counter(r.parent.parent for r in base.rglob("cache_report.json"))
        for root, n in roots.most_common():
            found.append((root, n, base.name))
    return found

def find_cache_zip():
    """A cache published as a zip. Kaggle archives large notebook outputs, so the
    dataset can be one 5 GB .zip rather than the directory tree."""
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if any(n.endswith("cache_report.json") for n in names):
            return z, names
    return None, []

def extract_cache(dest=Path("/kaggle/working/cache512")):
    """Extract a zipped cache once. Idempotent: a completed extraction is reused."""
    z, names = find_cache_zip()
    if z is None:
        return None
    marker = dest / ".extracted_from"
    if marker.exists() and marker.read_text().strip() == z.name:
        print(f"already extracted from {z.name} -> {dest}")
        return dest

    size_gb = z.stat().st_size / 2**30
    print(f"extracting {z.name} ({size_gb:.1f} GB, {len(names)} entries) -> {dest}")
    print("this takes a few minutes; it only happens once per session")
    dest.mkdir(parents=True, exist_ok=True)
    started = time.time()
    with zipfile.ZipFile(z) as zf:
        for i, name in enumerate(names, 1):
            zf.extract(name, dest)
            if i % 20000 == 0 or i == len(names):
                elapsed = time.time() - started
                rate = i / max(elapsed, 1e-9)
                print(f"  {i}/{len(names)}  {rate:.0f} files/s  "
                      f"eta {(len(names) - i) / max(rate, 1e-9) / 60:.1f} min", flush=True)
    marker.write_text(z.name)
    print(f"extracted in {(time.time() - started) / 60:.1f} min")
    return dest

def dataset_roots_by_name():
    """dataset -> the cache root that holds the best copy of it.

    The cache can be spread over more than one published dataset: a full build
    plus a later top-up. When a dataset appears in several roots, the one with
    the most mask channels wins, so a top-up that adds masks supersedes an
    earlier maskless copy.
    """
    best = {}
    for root, _, _ in cache_candidates():
        for d in sorted(p for p in root.iterdir() if p.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def find_cache_root(verbose=True):
    found = cache_candidates()
    if verbose and len(found) > 1:
        print("several cache roots found; using the first:")
        for root, n, base in found:
            print(f"   [{base}] {root}  ({n} datasets)")
    if found:
        return found[0][0]

    # Nothing unpacked anywhere - try a zipped cache before giving up.
    if extract_cache() is not None:
        found = cache_candidates()
        return found[0][0] if found else None
    return None

def run(cmd):
    """Run a build script, echoing it, and stop the notebook on failure."""
    print("$", cmd[:160] + (" ..." if len(cmd) > 160 else ""))
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(proc.stdout.rstrip())
    if proc.returncode != 0:
        print(proc.stderr.rstrip())
        raise RuntimeError(f"command failed with exit {proc.returncode}")
    return proc.stdout

def q(x):
    return shlex.quote(str(x))

from collections import Counter
print("cache helpers ready")

## 2 · Locate the cache

Works whether the cache arrives as a directory tree or as a **zip** — Kaggle archives
large notebook outputs, so a published cache is often one multi-GB `.zip`. A zip is
extracted once into `/kaggle/working` and reused on re-runs.

The cell also re-checks Phase 1's A0 gate, so nothing is built on a cache whose crops
went wrong.

In [ ]:
CACHE = find_cache_root()
print("cache root:", CACHE)
print()
assert CACHE is not None, (
    "no cache found. Add the verify-dr-cache-512 dataset (or the Phase 1 "
    "notebook's output) as an input. A zipped cache is handled automatically, "
    "but it must contain cache_report.json.")

# A dataset may live in a different root from the rest (a later top-up), so
# resolve each one independently.
ROOTS = dataset_roots_by_name()

# Re-confirm Phase 1's A0 gate before building anything on top of this cache.
print(f"{'dataset':<12}{'cached':>9}{'found':>9}{'fallback':>11}{'MiB':>9}  masks")
print("-" * 78)
a0 = True
for name in sorted(ROOTS):
    d = ROOTS[name] / name
    report = d / "cache_report.json"
    if not report.exists():
        continue
    r = json.loads(report.read_text())
    rate = r["crop"]["fallback_rate"]
    channels = sorted(c.name for c in (d / "masks").iterdir()) if (d / "masks").is_dir() else []
    ok = rate is not None and rate <= 0.005
    a0 &= ok
    print(f"{name:<12}{r.get('cached', 0):>9}{r['counts']['found']:>9}"
          f"{(f'{rate:.4f}' if rate is not None else 'n/a'):>11}"
          f"{r['output']['total_mib']:>9.0f}  {', '.join(channels) or '-'}"
          f"{'' if ok else '   <-- A0 gate 0.005 exceeded'}")

print()
if len({str(v) for v in ROOTS.values()}) > 1:
    print()
    print("cache spread over several roots:")
    for name in sorted(ROOTS):
        print(f"   {name:<12} {ROOTS[name]}")

print()
print("A0 (from Phase 1):", "PASS" if a0 else "REVIEW - see docs/04_experiment_register.md")
if not a0:
    print("A high crop fallback rate means many images kept their full frame instead")
    print("of being cropped to the retinal field. Check the Phase 1 contact sheets")
    print("before building manifests on this cache.")

## 3 · Locate the label files

Discovered rather than hardcoded, so a differently-organised mirror still works.

In [ ]:
# The cache has folders named ddr/eyepacs/idrid/aptos/messidor2; keep raw-dataset
# discovery out of it entirely.
EXCLUDE_ROOTS = [CACHE] + ([CACHE.parent] if CACHE.name != CACHE.parent.name else [])
print("excluding from raw discovery:", *EXCLUDE_ROOTS, sep="\n   ")
print()

ddr_raw   = find_mount("ddr", required=False)
idrid_raw = find_any(("idrid",), ("indian", "diabetic", "retinopathy"), required=False)
aptos_raw = find_mount("aptos", required=False)
m2_raw    = find_mount("messidor2", "grades", required=False) or find_mount("messidor", required=False)

def pick(root, *, suffixes, must=(), avoid=(), limit=None):
    if root is None:
        return []
    hits = []
    for f in root.rglob("*"):
        if not f.is_file() or f.suffix.lower() not in suffixes:
            continue
        low = str(f).lower()
        if any(a in low for a in avoid):
            continue
        if must and not all(any(m in low for m in group) for group in must):
            continue
        hits.append(f)
    hits = sorted(hits)
    return hits[:limit] if limit else hits

ddr_labels   = pick(ddr_raw, suffixes={".txt"}, must=[("grading",)])
idrid_grades = pick(idrid_raw, suffixes={".csv", ".xlsx"}, must=[("grad",)], avoid=("fovea", "centre", "center", "markup"))
idrid_coords = pick(idrid_raw, suffixes={".csv", ".xlsx"}, must=[("fovea", "od_", "centre", "center", "markup")])
aptos_labels = pick(aptos_raw, suffixes={".csv"}, must=[("train",)], avoid=("sample_sub",))
m2_labels    = pick(m2_raw, suffixes={".csv"}, avoid=("readme",))

idrid_originals = sorted(
    d for d in (idrid_raw.rglob("*") if idrid_raw else [])
    if d.is_dir() and "original" in str(d).lower() and "segmentation" in str(d).lower()
    and any(f.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif"} for f in d.iterdir() if f.is_file()))

for label, items in [("DDR .txt labels", ddr_labels), ("IDRiD Part B grades", idrid_grades),
                     ("IDRiD Part C coords", idrid_coords), ("IDRiD Part A originals", idrid_originals),
                     ("APTOS train.csv", aptos_labels), ("Messidor-2 grades", m2_labels)]:
    print(f"{label:24s} {len(items)}")
    for i in items[:4]:
        print("   ", i)

## 4 · EyePACS

Grade comes from a 0-4 class-folder component. If this mirror has no class folders the
cell says so and you will need a label CSV instead.

In [ ]:
eyepacs_imgs = ROOTS.get("eyepacs", CACHE) / "eyepacs" / "images"
has_class_dirs = any(d.is_dir() and d.name in {"0","1","2","3","4"}
                     for d in eyepacs_imgs.rglob("*"))
print("class folders present:", has_class_dirs)
assert has_class_dirs, (
    "no 0-4 class folders in the EyePACS cache -- grades must come from a label "
    "file instead; pass --labels rather than --folder-labels")

run(f"python {q(REPO_DIR/'scripts/prepare_manifest.py')} --dataset EyePACS "
    f"--cache-root {q(ROOTS.get('eyepacs', CACHE))} --folder-labels "
    f"--output {q(MANIFESTS/'eyepacs_manifest.csv')}")

## 5 · DDR

Grade-5 rows are ungradable and go to the quality manifest, not the grading one —
they are the supervision for the REACQUIRE head (experiment G2).

In [ ]:
assert ddr_labels, "no DDR DR_grading .txt files found - check the raw DDR mount"
label_args = " ".join(f"--labels {q(p)}" for p in ddr_labels)
run(f"python {q(REPO_DIR/'scripts/prepare_manifest.py')} --dataset DDR "
    f"--cache-root {q(ROOTS.get('ddr', CACHE))} {label_args} "
    f"--output {q(MANIFESTS/'ddr_manifest.csv')} "
    f"--quality-output {q(MANIFESTS/'quality_manifest.csv')}")

## 6 · IDRiD

Part C centres are published in **original pixel coordinates**, which the cache's crop,
square fit and resize invalidate. `prepare_manifest.py` recomputes each image's geometry
with the same functions `build_cache.py` used, so the two cannot drift — which is why
`--coords-source-dir` points at the raw originals, not the cache.

`--size`, `--fit` and `--tol-scale` must match the Phase 1 build.

In [ ]:
if idrid_grades:
    args = f"--labels {q(idrid_grades[0])}"
    if idrid_coords and idrid_originals:
        args += " " + " ".join(f"--coords {q(c)}" for c in idrid_coords)
        args += f" --coords-source-dir {q(idrid_originals[0])}"
    else:
        print("!! No Part C coordinates or Part A originals found.")
        print("!! C1 (optic-disc/fovea regressor) will have no training targets,")
        print("!! and M3 loses the coordinate frame its quadrant rules need.")
    run(f"python {q(REPO_DIR/'scripts/prepare_manifest.py')} --dataset IDRiD "
        f"--cache-root {q(ROOTS.get('idrid', CACHE))} {args} --size 512 --fit pad "
        f"--output {q(MANIFESTS/'idrid_manifest.csv')}")
else:
    print("!! No IDRiD grading table found - skipping. C2 will train on DDR masks alone.")

## 7 · APTOS and Messidor-2 — **locked**

Building their manifests is not evaluation. Nothing reads these until after the Phase 5
freeze (`docs/02_research_protocol.md` Rule 1).

In [ ]:
if aptos_labels:
    run(f"python {q(REPO_DIR/'scripts/prepare_manifest.py')} --dataset APTOS "
        f"--cache-root {q(ROOTS.get('aptos', CACHE))} --labels {q(aptos_labels[0])} "
        f"--output {q(MANIFESTS/'aptos_manifest.csv')}")
else:
    print("APTOS labels not found - skipping")

if m2_labels:
    run(f"python {q(REPO_DIR/'scripts/prepare_manifest.py')} --dataset Messidor2 "
        f"--cache-root {q(ROOTS.get('messidor2', CACHE))} --labels {q(m2_labels[0])} "
        f"--output {q(MANIFESTS/'messidor2_manifest.csv')}")
else:
    print("Messidor-2 grades not found - skipping")

## 8 · Assemble the variants

The three development variants share their validation, calibration and test rows
byte for byte; only training rows differ. That is what makes B6 and B7/H3 controlled
comparisons.

`--eyepacs-split regroup` builds a fresh patient-grouped split. Use `source` only if
this mirror genuinely ships the official competition split — and note Rule 6: a
regrouped test set is **not** leaderboard-comparable.

In [ ]:
opt = []
for flag, name in [("--ddr", "ddr"), ("--aptos", "aptos"), ("--messidor2", "messidor2")]:
    path = MANIFESTS / f"{name}_manifest.csv"
    if path.exists():
        opt.append(f"{flag} {q(path)}")

run(f"python {q(REPO_DIR/'scripts/build_variants.py')} "
    f"--eyepacs {q(MANIFESTS/'eyepacs_manifest.csv')} {' '.join(opt)} "
    f"--output-dir {q(MANIFESTS)} --seed 42 --eyepacs-split regroup")

## 9 · Verify independently

The build scripts assert these themselves. Checking again from the written CSVs is
cheap, and this is the rule everything else depends on.

In [ ]:
import pandas as pd

full = pd.read_csv(MANIFESTS / "eyepacs_full.csv")
# Variants only -- eyepacs_manifest.csv is the per-dataset manifest and has no split.
variants = {p.stem: pd.read_csv(p) for p in sorted(MANIFESTS.glob("eyepacs_*.csv"))
            if not p.name.endswith("_manifest.csv")}
assert "eyepacs_full" in variants, "build_variants.py did not write eyepacs_full.csv"

print("=== Rule 3: patient-grouped splits ===")
ok = True
for name, f in variants.items():
    bad = int((f.groupby("patient_id")["split"].nunique() > 1).sum())
    ok &= bad == 0
    print(f"  {name:24s} patients in >1 split: {bad}  {'OK' if bad == 0 else '*** LEAK ***'}")

print("\n=== eye pairing ===")
ratio = len(full) / max(1, full["patient_id"].nunique())
print(f"  EyePACS images/patient: {ratio:.2f}  (expect ~2.0)")
if ratio < 1.5:
    ok = False
    print("  *** patient parsing failed upstream - the split would leak ***")

print("\n=== Rule 4: held-out rows identical, test keeps natural prevalence ===")
held = {n: set(f.loc[f.split != "train", "image_path"]) for n, f in variants.items()}
base = held["eyepacs_full"]
for name, h in held.items():
    print(f"  {name:24s} held-out identical to full: {h == base}")
    ok &= h == base
for name, f in variants.items():
    te = f[f.split == "test"]["grade"].value_counts(normalize=True).sort_index() * 100
    tr = f[f.split == "train"]["grade"].value_counts(normalize=True).sort_index() * 100
    print(f"  {name:24s} train g0={tr.get(0, 0):5.1f}%   test g0={te.get(0, 0):5.1f}%")

print("\n=== Rule 1: externals locked ===")
for name in ("aptos_external", "messidor2_external"):
    p = MANIFESTS / f"{name}.csv"
    if p.exists():
        f = pd.read_csv(p)
        print(f"  {name:24s} locked={bool(f['locked'].all())} splits={set(f['split'])} rows={len(f)}")

print("\n" + ("PHASE 2 CHECKS PASSED" if ok else "*** CHECKS FAILED - do not continue ***"))

## 10 · The plan

In [ ]:
plan = json.loads((MANIFESTS / "dataset_plan.json").read_text())
print("split policy:", json.dumps(plan["eyepacs"]["split_policy"], indent=2))
for name, info in plan["variants"].items():
    print(f"\n{name}: {info['images']} images")
    for s in info["splits"]:
        print(f"   {s['split']:<12}{s['images']:>8} images  {s['patients']:>7} patients  {s['grades']}")
    if "balancing" in info and info["balancing"]["shortfall_per_grade"]:
        print(f"   shortfall: {info['balancing']['shortfall_per_grade']} (capped, never duplicated)")

---
## 11 · Save

**Save Version → Save & Run All (Commit).** Then publish `/kaggle/working/manifests`
as a Kaggle dataset named **`verify-dr-manifests`**, or add this notebook's output as
an input to Phase 3.

From here the raw datasets are no longer needed — Phase 3 reads the cache and these
manifests only.

> **Do not rebuild these manifests after training starts.** The split is part of every
> result. If you must, treat it as a protocol deviation and record it in
> `preregistration/PREREGISTRATION.md`.

Record A1 in `docs/04_experiment_register.md` once Phase 3's smoke test runs.